In [9]:
!nvidia-smi

Wed Jun  3 01:43:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
import os, subprocess

REPO_URL = "https://github.com/kriteenjain/COMSCI260c-project.git"
REPO_DIR = "/content/COMSCI260c-project"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls

cwd: /content/COMSCI260c-project
notebooks  requirements.txt  run_baseline.py	       run_qwen_compression.py
README.md  results	     run_llama_compression.py  src


In [11]:
!pip install -q -r requirements.txt
!pip install -q "autoawq>=0.2.6"
!pip install -q tabulate

In [12]:
from huggingface_hub import login, whoami
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    login(token=token)
except Exception:
    login()

os.environ["HF_TOKEN"] = token if 'token' in dir() else ""
os.environ["HF_HOME"] = "/content/.hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(whoami()["name"], "— authenticated ✓")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


seanhoffy — authenticated ✓


In [13]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
TASK = "both"
LIMIT = 100
BATCH_SIZE = 1                                    # 1 for LLaMA 3B on T4
COMPRESSED_DIR = "/content/compressed_llama"     # separate from Qwen cache

import os
os.makedirs(COMPRESSED_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)
print({"MODEL": MODEL, "TASK": TASK, "LIMIT": LIMIT, "BATCH_SIZE": BATCH_SIZE})

{'MODEL': 'meta-llama/Llama-3.2-3B-Instruct', 'TASK': 'both', 'LIMIT': 100, 'BATCH_SIZE': 1}


In [7]:
WANDA_24_DIR = f"{COMPRESSED_DIR}/llama-wanda-2-4"

!python run_llama_compression.py \
    --model {MODEL} \
    --method wanda \
    --sparsity-type 2:4 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {WANDA_24_DIR}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[load] meta-llama/Llama-3.2-3B-Instruct
config.json: 100% 878/878 [00:00<00:00, 3.70MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 84.6MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 51.2MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.48MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 20.9k/20.9k [00:00<00:00, 56.7MB/s]
Fetching 2 files: 100% 2/2 [00:16<00:00,  8.00s/it]
Download complete: 100% 6.43G/6.43G [00:16<00:00, 399MB/s]
Loading weights: 100% 254/254 [00:00<00:00, 938.70it/s, Material

In [15]:
!python run_llama_compression.py \
    --method wanda \
    --sparsity-type 2:4 \
    --task both \
    --limit 100 \
    --load-compressed /content/compressed_llama/llama-wanda-2-4 \
    --tag wanda-2-4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[load] pruned model from /content/compressed_llama/llama-wanda-2-4
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 254/254 [00:00<00:00, 822.83it/s, Materializing param=model.norm.weight]
[load] device=cuda dtype=torch.bfloat16 n_params=3,212,749,824
/content/COMSCI260c-project/run_llama_compression.py:182: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "started_at": dt.datetime.utcno